# Model selection and subject-share prediction

This notebook fits forecasting models for subject-share series and exports only the small set of forecast results needed by the plotting notebook.

The workflow is:

1. load original subject labels;
2. build pure- and applied-math share matrices for 1960–2024;
3. select a forecasting model by rolling-window cross-validation;
4. forecast subject shares for 2025–2027;
5. forecast the pure/applied basket ratio for 2025–2027;
6. save one compact CSV of forecast values.

Historical and provisional data are not exported because they can be reconstructed from `version2_new_dataset_fitted.csv` in the plotting notebook.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import pmdarima as pm
from pmdarima import auto_arima
from statsmodels.tsa.api import SimpleExpSmoothing, Holt
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 1. Configuration

In [2]:
DATA_PATH = Path("../../data/processed/version2_new_dataset_fitted.csv")
RESULTS_DIR = Path("outputs/subject_trend_predictions")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = RESULTS_DIR / "subject_prediction_forecasts.csv"

START_YEAR = 1960
END_YEAR = 2024
FORECAST_HORIZON = 3
WINDOW_SIZE = 20
CV_STEP = 3

# Keep this True to retain the four-model comparison from the current notebook.
# Set to False for a faster three-model comparison.
USE_AUTO_ARIMA = True

## 2. Load original subject-label data

In [13]:
nodes = pd.read_csv(DATA_PATH, low_memory=False)

nodes = nodes[
    nodes["subject_code"].notna()
    & nodes["year"].notna()
].copy()

#only keep the original data, and remove entries with predicted subject codes.
nodes = nodes[nodes["predicted_subject_code"].isna()].copy()

nodes["subject_code"] = pd.to_numeric(
    nodes["subject_code"],
    errors="coerce",
)

nodes["year"] = pd.to_numeric(
    nodes["year"],
    errors="coerce",
)

nodes = nodes.dropna(
    subset=["subject_code", "year"]
)

nodes["year"] = nodes["year"].astype(int)

print("Original-label records with subject and year:", f"{len(nodes):,}")

Original-label records with subject and year: 167,749


## 3. Build subject-share matrices

In [15]:
def build_subject_share_matrix(
    data,
    basket,
    start_year=START_YEAR,
    end_year=END_YEAR,
):
    years = np.arange(start_year, end_year + 1)

#subject code<=60 is pure math, and subject code>=62 is applied math
    if basket == "Pure":
        subset = data[
            data["subject_code"].lt(61)
            & data["year"].between(start_year, end_year)
        ].copy()
    elif basket == "Applied":
        subset = data[
            data["subject_code"].gt(61)
            & data["year"].between(start_year, end_year)
        ].copy()
    
    subject_codes = np.sort(subset["subject_code"].unique())

    counts = (
        pd.crosstab(subset["subject_code"], subset["year"])
        .reindex(index=subject_codes, columns=years, fill_value=0)
    )

    basket_total = counts.sum(axis=0).astype(float)
    share = counts.div(basket_total.replace(0, np.nan), axis=1) * 100

    return counts, share, basket_total


counts_pure, share_pure, pure_basket_total = (
    build_subject_share_matrix(nodes, "Pure")
)

counts_applied, share_applied, applied_basket_total = (
    build_subject_share_matrix(nodes, "Applied")
)

print("Pure share matrix:", share_pure.shape)
print("Applied share matrix:", share_applied.shape)

Pure share matrix: (44, 65)
Applied share matrix: (19, 65)


## 4. Forecasting models

In [5]:
def forecast_models(train, forecast_horizon):
    train = np.asarray(train, dtype=float)
    forecasts = {}

# 1. Simple Exponential Smoothing with alpha estimated inside the current training fold.
    ses = SimpleExpSmoothing(
        train,
        initialization_method="estimated",
    ).fit(optimized=True)

    forecasts["SES optimized"] = np.asarray(
        ses.forecast(forecast_horizon),
        dtype=float,
    )

# 2. Holt model (Double Exponential Smoothing), with parameters estimated inside the fold.
    holt = Holt(
        train,
        damped_trend=True,
        initialization_method="estimated",
    ).fit(optimized=True)

    forecasts["Holt damped"] = np.asarray(
        holt.forecast(forecast_horizon),
        dtype=float,
    )

# 3. Fixed ARIMA(0, 1, 1), retained as an existing benchmark.
    arima_011 = pm.ARIMA(
        order=(0, 1, 1),
        with_intercept=False,
        suppress_warnings=True,
    ).fit(train)

    forecasts["ARIMA(0,1,1)"] = np.asarray(
        arima_011.predict(n_periods=forecast_horizon),
        dtype=float,
    )

# 4. Auto-ARIMA.
    if USE_AUTO_ARIMA:
        auto_model = auto_arima(
            train,
            start_p=0,
            start_q=0,
            max_p=3,
            max_q=3,
            seasonal=False,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            trace=False,
        )

        forecasts["Auto ARIMA"] = np.asarray(
            auto_model.predict(n_periods=forecast_horizon),
            dtype=float,
        )

    return forecasts


def fit_and_forecast_model(y, model_name, forecast_horizon):
    y = np.asarray(y, dtype=float)

    if model_name == "SES optimized":
        model = SimpleExpSmoothing(
            y,
            initialization_method="estimated",
        ).fit(optimized=True)

        return np.asarray(
            model.forecast(forecast_horizon),
            dtype=float,
        )

    if model_name == "Holt damped":
        model = Holt(
            y,
            damped_trend=True,
            initialization_method="estimated",
        ).fit(optimized=True)

        return np.asarray(
            model.forecast(forecast_horizon),
            dtype=float,
        )

    if model_name == "ARIMA(0,1,1)":
        model = pm.ARIMA(
            order=(0, 1, 1),
            with_intercept=False,
            suppress_warnings=True,
        ).fit(y)

        return np.asarray(
            model.predict(n_periods=forecast_horizon),
            dtype=float,
        )

    if model_name == "Auto ARIMA":
        model = auto_arima(
            y,
            start_p=0,
            start_q=0,
            max_p=3,
            max_q=3,
            seasonal=False,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            trace=False,
        )

        return np.asarray(
            model.predict(n_periods=forecast_horizon),
            dtype=float,
        )

    raise ValueError(f"Unknown model: {model_name}")

## 5. Rolling-window cross-validation

In [6]:
def rolling_window_cv(
    y,
    window_size=WINDOW_SIZE,
    horizon=FORECAST_HORIZON,
    step=CV_STEP,
):
    y = np.asarray(y, dtype=float)

    records = []
    fold = 0

    for test_start in range(
        window_size,
        len(y) - horizon + 1,
        step,
    ):
        fold += 1

        train = y[test_start - window_size:test_start]
        test = y[test_start:test_start + horizon]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            forecasts = forecast_models(
                train=train,
                forecast_horizon=horizon,
            )

        mase_scale = np.mean(np.abs(np.diff(train)))

        for model_name, prediction in forecasts.items():
            mse = mean_squared_error(test, prediction)
            mae = mean_absolute_error(test, prediction)

            records.append(
                {
                    "fold": fold,
                    "model": model_name,
                    "MSE": mse,
                    "RMSE": np.sqrt(mse),
                    "MAE": mae,
                    "MASE": mae / mase_scale if mase_scale > 0 else np.nan,
                }
            )

    return pd.DataFrame(records)


def cross_validate_subject_matrix(matrix):
    all_results = []
    failures = []

    for row_position, subject_code in enumerate(matrix.index):
        y = matrix.loc[subject_code].astype(float).to_numpy()

        try:
            metrics = rolling_window_cv(y)
            metrics["row_position"] = row_position
            metrics["subject_code"] = subject_code
            all_results.append(metrics)

        except Exception as error:
            failures.append(
                {
                    "row_position": row_position,
                    "subject_code": subject_code,
                    "error": str(error),
                }
            )

    results = (
        pd.concat(all_results, ignore_index=True)
        if all_results
        else pd.DataFrame()
    )

    return results, pd.DataFrame(failures)


def select_model_by_subject_votes(cv_results, metric="MSE"):
    summary = (
        cv_results
        .groupby(["subject_code", "model"], as_index=False)
        .agg(mean_error=(metric, "mean"))
    )

    best_error = summary.groupby("subject_code")["mean_error"].transform("min")

    winners = summary[
        np.isclose(summary["mean_error"], best_error)
    ].copy()

    winners["number_of_ties"] = (
        winners
        .groupby("subject_code")["model"]
        .transform("count")
    )

    winners["vote"] = 1 / winners["number_of_ties"]

    vote_summary = (
        winners
        .groupby("model", as_index=False)
        .agg(
            total_votes=("vote", "sum"),
            subjects_with_win=("subject_code", "nunique"),
        )
        .sort_values(
            ["total_votes", "model"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    selected_model = vote_summary.loc[0, "model"]

    return selected_model, vote_summary, winners

## 6. Select models for pure and applied subject shares

In [7]:
pure_cv_results, pure_cv_failures = (
    cross_validate_subject_matrix(share_pure)
)

pure_share_selected_model, pure_vote_summary, pure_subject_votes = (
    select_model_by_subject_votes(pure_cv_results)
)

print("Pure-math selected model:", pure_share_selected_model)
display(pure_vote_summary)
print("Pure-math CV failures:", len(pure_cv_failures))

Pure-math selected model: Holt damped


,model,total_votes,subjects_with_win
0,Holt damped,19.0,19
1,"ARIMA(0,1,1)",13.0,13
2,SES optimized,9.0,9
3,Auto ARIMA,3.0,3


Pure-math CV failures: 0


In [8]:
applied_cv_results, applied_cv_failures = (
    cross_validate_subject_matrix(share_applied)
)

applied_share_selected_model, applied_vote_summary, applied_subject_votes = (
    select_model_by_subject_votes(applied_cv_results)
)

print("Applied-math selected model:", applied_share_selected_model)
display(applied_vote_summary)
print("Applied-math CV failures:", len(applied_cv_failures))

Applied-math selected model: ARIMA(0,1,1)


,model,total_votes,subjects_with_win
0,"ARIMA(0,1,1)",9.0,9
1,Holt damped,7.0,7
2,Auto ARIMA,2.0,2
3,SES optimized,1.0,1


Applied-math CV failures: 0


## 7. Forecast subject shares

In [9]:
def forecast_subject_shares(
    matrix,
    selected_model,
    basket,
    horizon=FORECAST_HORIZON,
    window_size=WINDOW_SIZE,
):
    train_matrix = matrix.iloc[:, -window_size:]
    future_years = np.arange(
        matrix.columns.max() + 1,
        matrix.columns.max() + horizon + 1,
    )

    forecast_rows = []

    for subject_code, row in train_matrix.iterrows():
        y = row.astype(float).to_numpy()

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            prediction = fit_and_forecast_model(
                y=y,
                model_name=selected_model,
                forecast_horizon=horizon,
            )

        prediction = np.clip(prediction, 0, None)

        for year, value in zip(future_years, prediction):
            forecast_rows.append(
                {
                    "record_type": "subject_share_forecast",
                    "basket": basket,
                    "subject_code": str(int(float(subject_code))).zfill(2),
                    "year": int(year),
                    "value_percent": float(value),
                    "model": selected_model,
                }
            )

    forecast = pd.DataFrame(forecast_rows)

    # Normalize each forecast year so shares within the basket sum to 100.
    totals = forecast.groupby("year")["value_percent"].transform("sum")
    forecast["value_percent"] = 100 * forecast["value_percent"] / totals

    return forecast


pure_subject_forecast = forecast_subject_shares(
    share_pure,
    selected_model=pure_share_selected_model,
    basket="Pure",
)

applied_subject_forecast = forecast_subject_shares(
    share_applied,
    selected_model=applied_share_selected_model,
    basket="Applied",
)

subject_forecasts = pd.concat(
    [pure_subject_forecast, applied_subject_forecast],
    ignore_index=True,
)

display(subject_forecasts.head())

,record_type,basket,subject_code,year,value_percent,model
0,subject_share_forecast,Pure,00,2025,0.916487,Holt damped
1,subject_share_forecast,Pure,00,2026,0.862613,Holt damped
2,subject_share_forecast,Pure,00,2027,0.808822,Holt damped
3,subject_share_forecast,Pure,01,2025,0.014166,Holt damped
4,subject_share_forecast,Pure,01,2026,0.000000,Holt damped


## 8. Forecast the pure/applied basket ratio

The historical pure-mathematics ratio is

$$
100\times
\frac{\text{pure-math records}}
{\text{pure-math records}+\text{applied-math records}}.
$$

This percentage series is forecast directly. The applied-mathematics ratio is then defined as $(100-\text{pure ratio})$, so the two forecasts sum to 100% by construction.

In [10]:
basket_totals = pd.concat(
    {
        "Pure": pure_basket_total,
        "Applied": applied_basket_total,
    },
    axis=1,
).dropna()

pure_ratio = (
    100
    * basket_totals["Pure"]
    / basket_totals.sum(axis=1)
)

applied_ratio = 100 - pure_ratio

pure_ratio_cv = rolling_window_cv(
    pure_ratio.to_numpy(dtype=float),
)

pure_ratio_cv_summary = (
    pure_ratio_cv
    .groupby("model", as_index=False)
    .agg(mean_MSE=("MSE", "mean"))
    .sort_values(["mean_MSE", "model"])
    .reset_index(drop=True)
)

pure_ratio_selected_model = pure_ratio_cv_summary.loc[0, "model"]

print("Pure-ratio selected model:", pure_ratio_selected_model)
display(pure_ratio_cv_summary)

Pure-ratio selected model: ARIMA(0,1,1)


,model,mean_MSE
0,"ARIMA(0,1,1)",5.647667
1,SES optimized,5.690110
2,Holt damped,5.884371
3,Auto ARIMA,7.036013


In [11]:
ratio_train = pure_ratio.iloc[-WINDOW_SIZE:].to_numpy(dtype=float)

ratio_prediction = fit_and_forecast_model(
    y=ratio_train,
    model_name=pure_ratio_selected_model,
    forecast_horizon=FORECAST_HORIZON,
)

future_years = np.arange(
    END_YEAR + 1,
    END_YEAR + FORECAST_HORIZON + 1,
)

pure_ratio_forecast = np.clip(ratio_prediction, 0, 100)
applied_ratio_forecast = 100 - pure_ratio_forecast

ratio_forecasts = pd.DataFrame(
    {
        "record_type": "basket_ratio_forecast",
        "basket": ["Pure"] * FORECAST_HORIZON
                  + ["Applied"] * FORECAST_HORIZON,
        "subject_code": [pd.NA] * (2 * FORECAST_HORIZON),
        "year": list(future_years) * 2,
        "value_percent": list(pure_ratio_forecast)
                         + list(applied_ratio_forecast),
        "model": [pure_ratio_selected_model] * FORECAST_HORIZON
                 + ["100 minus pure ratio"] * FORECAST_HORIZON,
    }
)

display(ratio_forecasts)

,record_type,basket,subject_code,year,value_percent,model
0,basket_ratio_forecast,Pure,<NA>,2025,48.734523,"ARIMA(0,1,1)"
1,basket_ratio_forecast,Pure,<NA>,2026,48.734523,"ARIMA(0,1,1)"
2,basket_ratio_forecast,Pure,<NA>,2027,48.734523,"ARIMA(0,1,1)"
3,basket_ratio_forecast,Applied,<NA>,2025,51.265477,100 minus pure ratio
4,basket_ratio_forecast,Applied,<NA>,2026,51.265477,100 minus pure ratio
5,basket_ratio_forecast,Applied,<NA>,2027,51.265477,100 minus pure ratio


## 9. Export forecast file

In [12]:
prediction_results = pd.concat(
    [
        subject_forecasts,
        ratio_forecasts,
    ],
    ignore_index=True,
)

prediction_results = prediction_results[
    [
        "record_type",
        "basket",
        "subject_code",
        "year",
        "value_percent",
        "model",
    ]
].sort_values(
    ["record_type", "basket", "subject_code", "year"],
    na_position="last",
)

prediction_results.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Saved compact forecast file:")
print(OUTPUT_PATH.resolve())
print("Rows:", len(prediction_results))
display(prediction_results.head())

Saved compact forecast file:
/Users/yimengliu/Documents/Erdos_Institute/mgp/summer26-math-genealogy/notebooks/subject_trend-Yilong/outputs/subject_trend_predictions/subject_prediction_forecasts.csv
Rows: 195


,record_type,basket,subject_code,year,value_percent,model
192,basket_ratio_forecast,Applied,<NA>,2025,51.265477,100 minus pure ratio
193,basket_ratio_forecast,Applied,<NA>,2026,51.265477,100 minus pure ratio
194,basket_ratio_forecast,Applied,<NA>,2027,51.265477,100 minus pure ratio
189,basket_ratio_forecast,Pure,<NA>,2025,48.734523,"ARIMA(0,1,1)"
190,basket_ratio_forecast,Pure,<NA>,2026,48.734523,"ARIMA(0,1,1)"
